In [1]:
# ============================================================================
# 1. Setup - participants, session branches, snapshots
# ============================================================================
# Every tool analyses the same per-branch snapshots, exported with `git archive`
# into .tools/snapshots/ - the clones under projects/ are never checked out.
# Tool runners (ESLint, Sonargraph) are shared with test.py.
import os, sys, re, time, subprocess
from pathlib import Path
import pandas as pd
import requests
from IPython.display import display

WORKSPACE = Path.cwd()
sys.path.insert(0, str(WORKSPACE))
import test as td

# participants -> session branches, in session order (edit here)
REPOS = {
    "aida-dpenner2": ["main", "s1", "s2", "s3", "s4"],
    "aida-makbar3":  ["main", "mujtaba/session-1", "mujtaba/session-2", "mujtaba/session-3", "mujtaba/session-4"],
}
td.REPOS = REPOS

REFRESH = False   # True = `git fetch` and re-export every snapshot

td.setup()                    # installs ESLint & friends into .tools/ on first run
td.RAW_DIR.mkdir(exist_ok=True)

def tag(repo, branch):
    return f"{repo}--{branch.split('/')[-1]}"

SNAPSHOTS, LOC = {}, {}
for repo, branches in REPOS.items():
    if REFRESH:
        subprocess.run(["git", "-C", str(td.PROJECTS / repo), "fetch", "-q", "origin"], check=True)
    for b in branches:
        root, _ = td.snapshot(repo, b, refresh=REFRESH)
        SNAPSHOTS[repo, b] = root
        LOC[repo, b] = sum(sum(1 for l in f.read_text(encoding="utf-8", errors="replace").splitlines()
                               if l.strip()) for f in td.code_files(root))

def compare(results, metrics, per_kloc=(), labels=None):
    """One table per participant: metrics as rows, session branches as columns.
    `per_kloc` metrics get an extra row divided by thousands of lines of code,
    since the code base grows several-fold across sessions."""
    labels = labels or {}
    for repo, branches in REPOS.items():
        t = pd.DataFrame({b: {m: results.get((repo, b), {}).get(m) for m in metrics} for b in branches})
        t.loc["lines of code"] = [LOC[repo, b] for b in branches]
        for m in per_kloc:
            t.loc[f"{labels.get(m, m)} / KLOC"] = (t.loc[m].astype(float) /
                                                    (t.loc["lines of code"] / 1000)).round(2)
        t = t.rename(index=labels)
        t.columns = [b.split("/")[-1] for b in t.columns]
        print(f"\n##### {repo}")
        display(t)

print(pd.DataFrame([{"repo": r, "branch": b, "lines of code": LOC[r, b]} for r, b in SNAPSHOTS])
        .pivot(index="repo", columns="branch", values="lines of code").fillna(""))


branch          main mujtaba/session-1 mujtaba/session-2 mujtaba/session-3  \
repo                                                                         
aida-dpenner2  354.0                                                         
aida-makbar3   354.0            1697.0            4416.0            7247.0   

branch        mujtaba/session-4      s1      s2      s3      s4  
repo                                                             
aida-dpenner2                    1260.0  3420.0  3729.0  6402.0  
aida-makbar3             8442.0                                  


In [2]:
# ============================================================================
# 2. SonarQube - issues, complexity and technical-debt estimate per session
# ============================================================================
# Needs Docker Desktop running. Starts the `sonarqube` container if it is
# stopped, reuses the token in .sonar-token, and scans each snapshot as its own
# project (Community Edition has no branch support).
SONAR_HOST      = "http://localhost:9000"
SONAR_CONTAINER = "sonarqube"
SONAR_ADMIN     = ("admin", "admin")
SCANNER_BIN     = next((td.TOOLS).glob("sonar-scanner-*/bin/sonar-scanner" + (".bat" if td.IS_WIN else "")))
SONAR_EXCLUSIONS = ",".join(td.EXCLUDE_GLOB + ["**/*.min.js", "**/*.lock", "**/*-lock.json", "**/*.map"])
SONAR_METRICS = ["ncloc", "complexity", "cognitive_complexity", "bugs", "vulnerabilities",
                 "code_smells", "security_hotspots", "sqale_index", "sqale_debt_ratio", "sqale_rating",
                 "reliability_rating", "security_rating", "duplicated_lines_density"]

# Docker Desktop installs per-user; a VS Code started before the install may not have it on PATH
_docker_bin = Path(os.environ.get("LOCALAPPDATA", "")) / "Programs/DockerDesktop/resources/bin"
if _docker_bin.exists() and str(_docker_bin) not in os.environ["PATH"]:
    os.environ["PATH"] += os.pathsep + str(_docker_bin)

def sonar_up():
    try:
        return requests.get(f"{SONAR_HOST}/api/system/status", timeout=5).json().get("status") == "UP"
    except Exception:
        return False

if not sonar_up():
    r = subprocess.run(["docker", "start", SONAR_CONTAINER], capture_output=True, text=True)
    if r.returncode:
        r = subprocess.run(["docker", "run", "-d", "--name", SONAR_CONTAINER, "-p", "9000:9000",
                            "sonarqube:community"], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"Could not start SonarQube - is Docker Desktop running?\n{r.stderr}")
    print("waiting for SonarQube ...", end="")
    for _ in range(120):
        if sonar_up():
            break
        time.sleep(5); print(".", end="", flush=True)
    print()

def api(method, path, **kw):
    return requests.request(method, f"{SONAR_HOST}{path}", auth=(SONAR_TOKEN, ""), timeout=60, **kw)

SONAR_TOKEN = td.WORKSPACE.joinpath(".sonar-token").read_text().strip() if (td.WORKSPACE / ".sonar-token").exists() else ""
if not SONAR_TOKEN or not api("GET", "/api/authentication/validate").json().get("valid"):
    r = requests.post(f"{SONAR_HOST}/api/user_tokens/generate", auth=SONAR_ADMIN, timeout=30,
                      data={"name": f"td-analysis-{int(time.time())}"})
    r.raise_for_status()
    SONAR_TOKEN = r.json()["token"]
    (td.WORKSPACE / ".sonar-token").write_text(SONAR_TOKEN)

def sonar_scan(repo, branch):
    key, root = tag(repo, branch), SNAPSHOTS[repo, branch]
    api("POST", "/api/projects/create", data={"project": key, "name": key})   # 400 = already exists
    r = subprocess.run([str(SCANNER_BIN), f"-Dsonar.projectKey={key}", f"-Dsonar.projectName={key}",
                        "-Dsonar.sources=apps,services" if (root / "services").exists() else "-Dsonar.sources=apps",
                        f"-Dsonar.host.url={SONAR_HOST}", f"-Dsonar.token={SONAR_TOKEN}",
                        f"-Dsonar.exclusions={SONAR_EXCLUSIONS}", "-Dsonar.sourceEncoding=UTF-8",
                        "-Dsonar.scm.disabled=true"],
                       cwd=root, capture_output=True, text=True, encoding="utf-8", errors="replace",
                       env=dict(os.environ, SONAR_SCANNER_OPTS="-Xmx2048m"))
    if r.returncode:
        raise RuntimeError(f"scanner failed for {key}:\n" + "\n".join(r.stdout.splitlines()[-30:]))
    meta = dict(l.split("=", 1) for l in (root / ".scannerwork/report-task.txt").read_text().splitlines() if "=" in l)
    for _ in range(100):                               # wait for background processing
        status = api("GET", "/api/ce/task", params={"id": meta["ceTaskId"]}).json()["task"]["status"]
        if status in ("SUCCESS", "FAILED", "CANCELED"):
            break
        time.sleep(3)
    if status != "SUCCESS":
        raise RuntimeError(f"SonarQube processing {status} for {key}")
    m = api("GET", "/api/measures/component",
            params={"component": key, "metricKeys": ",".join(SONAR_METRICS)}).json()["component"]["measures"]
    out = {x["metric"]: float(x["value"]) for x in m if "value" in x}
    facets = api("GET", "/api/issues/search", params={"componentKeys": key, "resolved": "false", "ps": 1,
                                                       "facets": "severities"}).json().get("facets", [])
    out.update({f"issues_{v['val'].lower()}": v["count"] for v in (facets[0]["values"] if facets else [])})
    return out

SONAR = {}
for repo, branches in REPOS.items():
    for b in branches:
        print("scanning", tag(repo, b))
        SONAR[repo, b] = sonar_scan(repo, b)

compare(SONAR,
        ["ncloc", "code_smells", "bugs", "vulnerabilities", "security_hotspots",
         "complexity", "cognitive_complexity", "sqale_index", "sqale_debt_ratio",
         "duplicated_lines_density",
         "issues_blocker", "issues_critical", "issues_major", "issues_minor", "issues_info"],
        per_kloc=["code_smells", "cognitive_complexity", "sqale_index"],
        labels={"ncloc": "Sonar lines of code", "code_smells": "code smells",
                "cognitive_complexity": "cognitive complexity", "sqale_index": "tech debt (min)",
                "sqale_debt_ratio": "tech debt ratio %",
                "duplicated_lines_density": "duplication %"})
# tech debt ratio = estimated fix time / estimated time to write the code (30 min
# per line), rounded to 0.1%. For these projects it is 0.0 everywhere - e.g. 40 min
# of debt against ~196,000 min of effort is 0.02% - so it is left out of the summary;
# "tech debt (min) / KLOC" carries the same information at a readable scale.


waiting for SonarQube .......
scanning aida-dpenner2--main
scanning aida-dpenner2--s1
scanning aida-dpenner2--s2
scanning aida-dpenner2--s3
scanning aida-dpenner2--s4
scanning aida-makbar3--main
scanning aida-makbar3--session-1
scanning aida-makbar3--session-2
scanning aida-makbar3--session-3
scanning aida-makbar3--session-4

##### aida-dpenner2


,main,s1,s2,s3,s4
Sonar lines of code,608.00,1545.00,3695.00,3966.00,6547.00
code smells,0.00,0.00,1.00,1.00,5.00
bugs,0.00,0.00,4.00,4.00,7.00
vulnerabilities,1.00,1.00,1.00,1.00,1.00
security_hotspots,0.00,0.00,0.00,0.00,0.00
complexity,41.00,226.00,605.00,684.00,1156.00
cognitive complexity,14.00,112.00,295.00,422.00,645.00
tech debt (min),0.00,0.00,5.00,5.00,40.00
tech debt ratio %,0.00,0.00,0.00,0.00,0.00
duplication %,0.00,0.00,0.90,0.80,1.80



##### aida-makbar3


,main,session-1,session-2,session-3,session-4
Sonar lines of code,608.00,1776.00,4451.00,7077.00,8139.00
code smells,0.00,1.00,2.00,2.00,3.00
bugs,0.00,0.00,0.00,0.00,0.00
vulnerabilities,1.00,0.00,0.00,0.00,0.00
security_hotspots,0.00,0.00,0.00,0.00,0.00
complexity,41.00,274.00,666.00,1146.00,1308.00
cognitive complexity,14.00,151.00,285.00,596.00,671.00
tech debt (min),0.00,5.00,25.00,25.00,45.00
tech debt ratio %,0.00,0.00,0.00,0.00,0.00
duplication %,0.00,0.00,1.50,1.50,2.40


In [3]:
# ============================================================================
# 3. ESLint - JavaScript code smells (Johannes et al., SQJ 2019)
# ============================================================================
# 9 of the paper's 12 smells, mapped onto built-in ESLint rules (see
# td.ESLINT_SMELLS in test.py). Thresholds are ESLint's defaults - change them
# there to match the paper before reporting.
#
# Also measures how concentrated the complexity is: the cyclomatic complexity
# of every function (ESLint `complexity` rule), and the share above
# td.COMPLEX_FN_THRESHOLD - debt sits in a few complex functions, which
# per-file or per-KLOC averages hide.
ESL = {}
for repo, branches in REPOS.items():
    for b in branches:
        ESL[repo, b] = {**td.eslint(SNAPSHOTS[repo, b], tag(repo, b), None),
                        **td.eslint_function_complexity(SNAPSHOTS[repo, b], tag(repo, b), None)}

smells = [f"eslint_{s}" for s in td.ESLINT_SMELLS]
compare(ESL, ["eslint_smells"] + smells, per_kloc=["eslint_smells"],
        labels={"eslint_smells": "all smells",
                **{f"eslint_{s}": s.replace("_", " ") for s in td.ESLINT_SMELLS}})

print(f"\n=== Function complexity (cyclomatic, threshold {td.COMPLEX_FN_THRESHOLD}) ===")
compare(ESL, ["fn_count", "fn_over_threshold", "fn_pct_over_threshold", "fn_ccn_p90", "fn_ccn_max"],
        labels={"fn_count": "functions", "fn_over_threshold": f"functions over {td.COMPLEX_FN_THRESHOLD}",
                "fn_pct_over_threshold": f"% functions over {td.COMPLEX_FN_THRESHOLD}",
                "fn_ccn_p90": "90th percentile complexity", "fn_ccn_max": "max complexity"})

# the worst offenders on each participant's latest session
import json
for repo, branches in REPOS.items():
    raw = json.loads((td.RAW_DIR / f"eslint-{tag(repo, branches[-1])}.json").read_text(encoding="utf-8"))
    rows = [{"file": f["filePath"].replace("\\", "/").split(f"{tag(repo, branches[-1])}/")[-1],
             "line": m["line"], "smell": td.RULE_TO_SMELL.get(m.get("ruleId")), "message": m["message"]}
            for f in raw for m in f["messages"] if not m.get("fatal")]
    print(f"\n##### {repo} - {branches[-1]}: {len(rows)} smells")
    display(pd.DataFrame(rows).sort_values(["smell", "file", "line"]).reset_index(drop=True))



##### aida-dpenner2


,main,s1,s2,s3,s4
all smells,0.0,6.00,19.00,19.0,35.00
lengthy lines,0.0,0.00,1.00,1.0,7.00
long parameter list,0.0,0.00,1.00,1.0,5.00
nested callbacks,0.0,0.00,0.00,0.0,0.00
complex code,0.0,2.00,7.00,7.0,9.00
depth,0.0,0.00,1.00,1.0,0.00
long method,0.0,4.00,9.00,9.0,14.00
assign in condition,0.0,0.00,0.00,0.0,0.00
param reassign,0.0,0.00,0.00,0.0,0.00
extra bind,0.0,0.00,0.00,0.0,0.00



##### aida-makbar3


,main,session-1,session-2,session-3,session-4
all smells,0.0,7.00,13.00,22.00,25.00
lengthy lines,0.0,0.00,0.00,0.00,0.00
long parameter list,0.0,0.00,1.00,1.00,2.00
nested callbacks,0.0,0.00,0.00,0.00,0.00
complex code,0.0,3.00,6.00,11.00,12.00
depth,0.0,1.00,1.00,1.00,1.00
long method,0.0,3.00,5.00,9.00,10.00
assign in condition,0.0,0.00,0.00,0.00,0.00
param reassign,0.0,0.00,0.00,0.00,0.00
extra bind,0.0,0.00,0.00,0.00,0.00



=== Function complexity (cyclomatic, threshold 10) ===

##### aida-dpenner2


,main,s1,s2,s3,s4
functions,19.0,112.0,286.0,286.0,497.0
functions over 10,0.0,2.0,7.0,7.0,9.0
% functions over 10,0.0,1.8,2.4,2.4,1.8
90th percentile complexity,6.0,3.0,5.0,5.0,5.0
max complexity,7.0,19.0,19.0,19.0,23.0
lines of code,354.0,1260.0,3420.0,3729.0,6402.0


... (2 outputs left out)


##### aida-dpenner2 - s4: 35 smells


,file,line,smell,message
0,apps/api/src/routes/conversations.ts,30,complex_code,Arrow function has a complexity of 11. Maximum...
1,apps/api/src/routes/conversations.ts,81,complex_code,Async arrow function has a complexity of 23. M...
2,apps/api/src/routes/practice.ts,16,complex_code,Async arrow function has a complexity of 11. M...
3,apps/web/src/components/CodeStepper.tsx,32,complex_code,Function 'CodeStepper' has a complexity of 13....
4,apps/web/src/lib/api.ts,152,complex_code,Async function 'streamAssistantMessage' has a ...
5,apps/web/src/pages/HomePage.tsx,7,complex_code,Function 'HomePage' has a complexity of 13. Ma...
6,apps/web/src/pages/VerifyEmailPage.tsx,6,complex_code,Function 'VerifyEmailPage' has a complexity of...
7,apps/web/src/pages/instructor/CourseDetailPage...,42,complex_code,Function 'CourseDetailPage' has a complexity o...
8,apps/web/src/pages/instructor/CoursesPage.tsx,8,complex_code,Function 'CoursesPage' has a complexity of 11....
9,apps/api/src/db/materials.ts,80,lengthy_lines,This line has a length of 121. Maximum allowed...



##### aida-makbar3 - mujtaba/session-4: 25 smells


,file,line,smell,message
0,apps/api/src/services/crawlerService.ts,59,complex_code,Async function 'crawlSite' has a complexity of...
1,apps/web/src/components/ChatView.tsx,16,complex_code,Function 'ChatView' has a complexity of 15. Ma...
2,apps/web/src/components/ConversationSidebar.tsx,8,complex_code,Function 'ConversationSidebar' has a complexit...
3,apps/web/src/components/PracticeQuestionBlock.tsx,18,complex_code,Function 'PracticeQuestionBlock' has a complex...
4,apps/web/src/components/PythonCodeBlock.tsx,17,complex_code,Function 'PythonCodeBlock' has a complexity of...
5,apps/web/src/components/TraceViewer.tsx,98,complex_code,Function 'VariableValue' has a complexity of 1...
6,apps/web/src/hooks/useChatStream.ts,69,complex_code,Async arrow function has a complexity of 16. M...
7,apps/web/src/lib/apiClient.ts,13,complex_code,Async function 'apiFetch' has a complexity of ...
8,apps/web/src/routes/InstructorCourseDetailPage...,191,complex_code,Function 'MaterialsSection' has a complexity o...
9,apps/web/src/routes/InstructorCourseDetailPage...,824,complex_code,Function 'EnrollmentSection' has a complexity ...


In [4]:
# ============================================================================
# 4. Sonargraph - architectural debt (cycles, coupling, structural debt)
# ============================================================================
# Needs .sonargraph-activation (Build activation code, gitignored) and the
# TypeScript system in .tools/sonargraph-template/ - see td.sonargraph().
#   ACD  - average number of files each file depends on (Lakos)
#   NCCD - ACD relative to a balanced binary tree; > 1 = more tangled than a tree
#   propagation cost - share of file pairs a change can reach (MacCormack et al.);
#                      falls as the code base grows, so compare at similar sizes
SG = {}
for repo, branches in REPOS.items():
    for b in branches:
        print("analysing", tag(repo, b))
        SG[repo, b] = td.sonargraph(SNAPSHOTS[repo, b], tag(repo, b), None)

compare(SG,
        ["sg_Number of Source Files", "sg_ACD", "sg_NCCD", "sg_Propagation Cost",
         "sg_Number of Component Cycle Groups", "sg_Number of Cyclic Components",
         "sg_Structural Debt Index (Components)",
         "sg_Biggest Typescript Directory Cycle Group", "sg_Structural Debt Index (Typescript Directories)",
         "sg_Relative Entanglement (%)", "sg_Average Complexity", "sg_Average Block Nesting Depth"],
        per_kloc=["sg_Structural Debt Index (Typescript Directories)"],
        labels={"sg_Number of Source Files": "files", "sg_ACD": "ACD", "sg_NCCD": "NCCD",
                "sg_Propagation Cost": "propagation cost %",
                "sg_Number of Component Cycle Groups": "file cycle groups",
                "sg_Number of Cyclic Components": "files in cycles",
                "sg_Structural Debt Index (Components)": "structural debt (files)",
                "sg_Biggest Typescript Directory Cycle Group": "biggest directory cycle",
                "sg_Structural Debt Index (Typescript Directories)": "structural debt (directories)",
                "sg_Relative Entanglement (%)": "relative entanglement %",
                "sg_Average Complexity": "average complexity",
                "sg_Average Block Nesting Depth": "average nesting depth"})


analysing aida-dpenner2--main
analysing aida-dpenner2--s1
analysing aida-dpenner2--s2
analysing aida-dpenner2--s3
analysing aida-dpenner2--s4
analysing aida-makbar3--main
analysing aida-makbar3--session-1
analysing aida-makbar3--session-2
analysing aida-makbar3--session-3
analysing aida-makbar3--session-4

##### aida-dpenner2


,main,s1,s2,s3,s4
files,6.00,24.00,41.00,41.00,71.00
ACD,1.17,3.54,5.34,5.34,7.70
NCCD,0.51,0.92,1.18,1.18,1.47
propagation cost %,19.44,14.76,13.03,13.03,10.85
file cycle groups,0.00,0.00,0.00,0.00,1.00
files in cycles,0.00,0.00,0.00,0.00,2.00
structural debt (files),0.00,0.00,0.00,0.00,12.00
biggest directory cycle,0.00,4.00,5.00,5.00,5.00
structural debt (directories),0.00,24.00,50.00,50.00,74.00
relative entanglement %,0.00,13.33,15.38,15.38,15.02



##### aida-makbar3


,main,session-1,session-2,session-3,session-4
files,6.00,45.00,82.00,106.00,115.00
ACD,1.17,4.53,6.09,6.73,7.09
NCCD,0.51,0.98,1.12,1.16,1.20
propagation cost %,19.44,10.07,7.42,6.35,6.16
file cycle groups,0.00,0.00,0.00,0.00,0.00
files in cycles,0.00,0.00,0.00,0.00,0.00
structural debt (files),0.00,0.00,0.00,0.00,0.00
biggest directory cycle,0.00,7.00,9.00,9.00,9.00
structural debt (directories),0.00,50.00,149.00,237.00,266.00
relative entanglement %,0.00,17.50,20.00,20.00,20.00


In [5]:
# ============================================================================
# 5. Summary - technical debt per session, all three tools
# ============================================================================
# Only size-independent metrics (ratios, percentages, per-KLOC), since the code
# base grows several-fold across sessions and raw counts mostly measure growth.
# `main` is the starter template every participant forked - the baseline.
#
#   tech debt min / KLOC   SonarQube: remediation minutes per 1000 lines
#   cognitive cx / KLOC    SonarQube: how hard the code is to understand
#   smells / KLOC          ESLint: Johannes et al. JavaScript smells
#   % complex functions    ESLint: share of functions with cyclomatic complexity > 10
#   NCCD                   Sonargraph: coupling vs. a balanced tree (> 1 = more tangled)
#   files in cycles        Sonargraph: files in circular dependencies
#   structural debt / KLOC Sonargraph: effort to break directory-level cycles
def per_kloc(v, repo, b):
    return None if v is None else round(v / (LOC[repo, b] / 1000), 2)

FINAL = [   # (tool, label, value function)
    ("",           "lines of code",          lambda r, b: LOC[r, b]),
    ("SonarQube",  "tech debt min / KLOC",   lambda r, b: per_kloc(SONAR[r, b].get("sqale_index"), r, b)),
    ("SonarQube",  "cognitive cx / KLOC",    lambda r, b: per_kloc(SONAR[r, b].get("cognitive_complexity"), r, b)),
    ("ESLint",     "smells / KLOC",          lambda r, b: per_kloc(ESL[r, b].get("eslint_smells"), r, b)),
    ("ESLint",     f"% functions cx > {td.COMPLEX_FN_THRESHOLD}", lambda r, b: ESL[r, b].get("fn_pct_over_threshold")),
    ("Sonargraph", "NCCD",                   lambda r, b: SG[r, b].get("sg_NCCD")),
    ("Sonargraph", "files in cycles",        lambda r, b: SG[r, b].get("sg_Number of Cyclic Components")),
    ("Sonargraph", "structural debt / KLOC", lambda r, b: per_kloc(SG[r, b].get("sg_Structural Debt Index (Typescript Directories)"), r, b)),
]

summary = {}
for repo, branches in REPOS.items():
    t = pd.DataFrame({b.split("/")[-1]: [f(repo, b) for _, _, f in FINAL] for b in branches},
                     index=pd.MultiIndex.from_tuples([(tool, label) for tool, label, _ in FINAL]))
    sessions = t.columns[1:]                          # everything after the `main` baseline
    t[f"change {sessions[0]} -> {sessions[-1]}"] = (t[sessions[-1]] - t[sessions[0]]).round(2)
    summary[repo] = t
    print(f"\n##### {repo}  (main = starter template baseline)")
    display(t)
    t.to_csv(td.RESULTS / repo / "td_summary.csv")

# both participants' latest session side by side
latest = pd.DataFrame({f"{repo} ({branches[-1].split('/')[-1]})": summary[repo].iloc[:, -2]
                       for repo, branches in REPOS.items()})
print("\n##### latest session, side by side")
display(latest)
latest.to_csv(td.RESULTS / "td_summary_latest.csv")
print(f"saved results/<repo>/td_summary.csv and results/td_summary_latest.csv")



##### aida-dpenner2  (main = starter template baseline)


main       s1       s2       s3       s4  \
           lines of code           354.00  1260.00  3420.00  3729.00  6402.00   
SonarQube  tech debt min / KLOC      0.00     0.00     1.46     1.34     6.25   
           cognitive cx / KLOC      39.55    88.89    86.26   113.17   100.75   
ESLint     smells / KLOC             0.00     4.76     5.56     5.10     5.47   
           % functions cx > 10       0.00     1.80     2.40     2.40     1.80   
Sonargraph NCCD                      0.51     0.92     1.18     1.18     1.47   
           files in cycles           0.00     0.00     0.00     0.00     2.00   
           structural debt / KLOC    0.00    19.05    14.62    13.41    11.56   

                                   change s1 -> s4  
           lines of code                   5142.00  
SonarQube  tech debt min / KLOC               6.25  
           cognitive cx / KLOC               11.86  
ESLint     smells / KLOC                      0.71  
           % functions cx > 10                0.00  
Sonargraph NCCD                               0.55  
           files in cycles                    2.00  
           structural debt / KLOC            -7.49


##### aida-makbar3  (main = starter template baseline)


main  session-1  session-2  session-3  \
           lines of code           354.00    1697.00    4416.00    7247.00   
SonarQube  tech debt min / KLOC      0.00       2.95       5.66       3.45   
           cognitive cx / KLOC      39.55      88.98      64.54      82.24   
ESLint     smells / KLOC             0.00       4.12       2.94       3.04   
           % functions cx > 10       0.00       2.40       1.70       1.90   
Sonargraph NCCD                      0.51       0.98       1.12       1.16   
           files in cycles           0.00       0.00       0.00       0.00   
           structural debt / KLOC    0.00      29.46      33.74      32.70   

                                   session-4  change session-1 -> session-4  
           lines of code             8442.00                        6745.00  
SonarQube  tech debt min / KLOC         5.33                           2.38  
           cognitive cx / KLOC         79.48                          -9.50  
ESLint     smells / KLOC                2.96                          -1.16  
           % functions cx > 10          1.80                          -0.60  
Sonargraph NCCD                         1.20                           0.22  
           files in cycles              0.00                           0.00  
           structural debt / KLOC      31.51                           2.05


##### latest session, side by side


aida-dpenner2 (s4)  \
           lines of code                      6402.00   
SonarQube  tech debt min / KLOC                  6.25   
           cognitive cx / KLOC                 100.75   
ESLint     smells / KLOC                         5.47   
           % functions cx > 10                   1.80   
Sonargraph NCCD                                  1.47   
           files in cycles                       2.00   
           structural debt / KLOC               11.56   

                                   aida-makbar3 (session-4)  
           lines of code                            8442.00  
SonarQube  tech debt min / KLOC                        5.33  
           cognitive cx / KLOC                        79.48  
ESLint     smells / KLOC                               2.96  
           % functions cx > 10                         1.80  
Sonargraph NCCD                                        1.20  
           files in cycles                             0.00  
           structural debt / KLOC                     31.51

saved results/<repo>/td_summary.csv and results/td_summary_latest.csv
